In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import shutil, os
shutil.copytree("/content/drive/MyDrive/60_pic", "/content/60_pic")

'/content/60_pic'

In [3]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
import matplotlib.pyplot as plt
import pickle

In [4]:
train_dir = "/content/60_pic"
img_size = 224
batch_size = 32

In [5]:
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=15,
    brightness_range=[0.7, 1.3],
    zoom_range=0.1,
    horizontal_flip=True,
    validation_split=0.2)

validation_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    validation_split=0.2)

In [6]:
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(img_size, img_size),
    batch_size=batch_size,
    class_mode='categorical',
    subset='training',
    shuffle=True)

validation_generator = validation_datagen.flow_from_directory(
    train_dir,
    target_size=(img_size, img_size),
    batch_size=batch_size,
    class_mode='categorical',
    subset='validation',
    shuffle=False)

Found 1570 images belonging to 31 classes.
Found 387 images belonging to 31 classes.


In [7]:
num_classes = train_generator.num_classes
print(f"Classes: {num_classes}, Train: {train_generator.n}, Val: {validation_generator.n}")
with open("/content/drive/MyDrive/class_names.pkl", "wb") as f:
    pickle.dump(train_generator.class_indices, f)

Classes: 31, Train: 1570, Val: 387


In [8]:
base = MobileNetV2(weights='imagenet',
                   include_top=False,
                   input_shape=(img_size, img_size, 3))
base.trainable = False

x = base.output
x = GlobalAveragePooling2D()(x) #đưa x đi qua layer này
x = BatchNormalization()(x)
x = Dropout(0.4)(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.3)(x)
output = Dense(num_classes, activation='softmax')(x)

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [9]:
model = Model(inputs=base.input, outputs=output)
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 112, 112,  │        864 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 112, 112,  │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 112, 112,  │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 112, 112,  │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 112, 112,  │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 112, 112,  │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 113, 113,  │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 56, 56,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │          0 │ block_1_depthwis… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 56, 56,    │      2,304 │ block_1_depthwis

 Total params: 2,599,007 (9.91 MB)

 Trainable params: 338,463 (1.29 MB)

 Non-trainable params: 2,260,544 (8.62 MB)

In [10]:
callbacks = [
    ModelCheckpoint("/content/drive/MyDrive/best_model.h5", monitor='val_accuracy', save_best_only=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6),
    EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True)
]

history1 = model.fit(train_generator, validation_data=validation_generator,
                     epochs=10, callbacks=callbacks)

Epoch 1/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.2423 - loss: 3.2859
Epoch 1: val_accuracy improved from None to 0.56589, saving model to /content/drive/MyDrive/best_model.h5



Epoch 1: finished saving model to /content/drive/MyDrive/best_model.h5
50/50 ━━━━━━━━━━━━━━━━━━━━ 163s 3s/step - accuracy: 0.4223 - loss: 2.3065 - val_accuracy: 0.5659 - val_loss: 1.7735 - learning_rate: 0.0010
Epoch 2/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.7564 - loss: 0.8250
Epoch 2: val_accuracy improved from 0.56589 to 0.70284, saving model to /content/drive/MyDrive/best_model.h5



Epoch 2: finished saving model to /content/drive/MyDrive/best_model.h5
50/50 ━━━━━━━━━━━━━━━━━━━━ 103s 2s/step - accuracy: 0.7745 - loss: 0.7576 - val_accuracy: 0.7028 - val_loss: 1.2954 - learning_rate: 0.0010
Epoch 3/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.8500 - loss: 0.4925
Epoch 3: val_accuracy improved from 0.70284 to 0.71835, saving model to /content/drive/MyDrive/best_model.h5



Epoch 3: finished saving model to /content/drive/MyDrive/best_model.h5
50/50 ━━━━━━━━━━━━━━━━━━━━ 101s 2s/step - accuracy: 0.8618 - loss: 0.4548 - val_accuracy: 0.7183 - val_loss: 1.0955 - learning_rate: 0.0010
Epoch 4/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.8739 - loss: 0.4559
Epoch 4: val_accuracy improved from 0.71835 to 0.72093, saving model to /content/drive/MyDrive/best_model.h5



Epoch 4: finished saving model to /content/drive/MyDrive/best_model.h5
50/50 ━━━━━━━━━━━━━━━━━━━━ 99s 2s/step - accuracy: 0.8866 - loss: 0.3963 - val_accuracy: 0.7209 - val_loss: 1.0292 - learning_rate: 0.0010
Epoch 5/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.8971 - loss: 0.3401
Epoch 5: val_accuracy improved from 0.72093 to 0.73643, saving model to /content/drive/MyDrive/best_model.h5



Epoch 5: finished saving model to /content/drive/MyDrive/best_model.h5
50/50 ━━━━━━━━━━━━━━━━━━━━ 100s 2s/step - accuracy: 0.9025 - loss: 0.3197 - val_accuracy: 0.7364 - val_loss: 1.0032 - learning_rate: 0.0010
Epoch 6/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9188 - loss: 0.2679
Epoch 6: val_accuracy improved from 0.73643 to 0.75711, saving model to /content/drive/MyDrive/best_model.h5



Epoch 6: finished saving model to /content/drive/MyDrive/best_model.h5
50/50 ━━━━━━━━━━━━━━━━━━━━ 101s 2s/step - accuracy: 0.9146 - loss: 0.2862 - val_accuracy: 0.7571 - val_loss: 0.9326 - learning_rate: 0.0010
Epoch 7/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9304 - loss: 0.2224
Epoch 7: val_accuracy improved from 0.75711 to 0.76227, saving model to /content/drive/MyDrive/best_model.h5



Epoch 7: finished saving model to /content/drive/MyDrive/best_model.h5
50/50 ━━━━━━━━━━━━━━━━━━━━ 102s 2s/step - accuracy: 0.9312 - loss: 0.2185 - val_accuracy: 0.7623 - val_loss: 0.9551 - learning_rate: 0.0010
Epoch 8/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9387 - loss: 0.1988
Epoch 8: val_accuracy did not improve from 0.76227
50/50 ━━━━━━━━━━━━━━━━━━━━ 99s 2s/step - accuracy: 0.9338 - loss: 0.2198 - val_accuracy: 0.7442 - val_loss: 0.9911 - learning_rate: 0.0010
Epoch 9/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9470 - loss: 0.1763
Epoch 9: val_accuracy did not improve from 0.76227
50/50 ━━━━━━━━━━━━━━━━━━━━ 100s 2s/step - accuracy: 0.9465 - loss: 0.1690 - val_accuracy: 0.7519 - val_loss: 1.0259 - learning_rate: 0.0010
Epoch 10/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9540 - loss: 0.1490
Epoch 10: val_accuracy did not improve from 0.76227
50/50 ━━━━━━━━━━━━━━━━━━━━ 98s 2s/step - accuracy: 0.9529 - loss: 0.1454 - val_accuracy: 0.7183 - val_los

In [11]:
val_loss, val_acc = model.evaluate(validation_generator)
print(f"Final Val Accuracy: {val_acc:.4f}")

13/13 ━━━━━━━━━━━━━━━━━━━━ 16s 1s/step - accuracy: 0.7623 - loss: 0.9551
Final Val Accuracy: 0.7623
